In [36]:
import random
import json
import time
import os
import uuid

from datetime import datetime, timezone
from google import genai
from getpass import getpass

In [2]:
# configuring the gemini client

if not os.environ.get('GEMINI_API_KEY'):
    os.environ['GEMINI_API_KEY']= getpass.getpass('Enter Gemini API key: ')

client = genai.Client()
print('Gemini client initialized successfully!')

Enter Gemini API key:  ········


Gemini client initialized successfully!


In [3]:
MODEL_NAME = "gemini-3.5-flash-lite"
print('Selected model: ', MODEL_NAME)

Selected model:  gemini-3.5-flash-lite


In [4]:
#defining the game configuration

PUZZLE_CATEGORIES = [
    'number_pattern',
    'riddle',
    'logic'
]

DIFFICULTY_LEVELS = [
    'easy',
    'mediium',
    'hard'
]

STARTING_DIFFICULTY = 'easy'

MAX_HINTS = 3
RECENT_HISTORY_LIMIT = 10

print('Puzzle categories:', PUZZLE_CATEGORIES)
print('Difficulty levels:', DIFFICULTY_LEVELS)
print('Starting difficulty:', STARTING_DIFFICULTY)

Puzzle categories: ['number_pattern', 'riddle', 'logic']
Difficulty levels: ['easy', 'mediium', 'hard']
Starting difficulty: easy


In [5]:
#defining game session

def create_game_session():
    session = {
        'session_id': f"session_{int(time.time())}",
        
        'current_difficulty': STARTING_DIFFICULTY,
        
        'total_attempts': 0,
        'correct_answers': 0,
        'incorrect_answers': 0,

        'total_time': 0,
        'total_hints_used':0,

        'attempt_history': [],
        'puzzle_history': [],
        'adaptation_history': []
    }

    return session

game_session = create_game_session()
print(game_session)

{'session_id': 'session_1795030871', 'current_difficulty': 'easy', 'total_attempts': 0, 'correct_answers': 0, 'incorrect_answers': 0, 'total_time': 0, 'total_hints_used': 0, 'attempt_history': [], 'puzzle_history': [], 'adaptation_history': []}


In [6]:
#defining a random category selector

def select_random_category(session):
    previous_category = None

    if session['puzzle_history']:
        previous_category = session['puzzle_history'][-1]['category']

    available_categories = [
        category
        for category in PUZZLE_CATEGORIES
        if category != previous_category
    ]

    selected_category = random.choice(available_categories)

    return selected_category

In [7]:
#defining a common puzzle structure for all categories

def create_puzzle(
    category,
    difficulty,
    question,
    content,
    answer_type,
    correct_answer,
    hints,
    explanation,
    options=None,
    accepted_answers=None,
    source='ai'
):
    puzzle={
        'puzzle_id':str(uuid.uuid4()),

        #Common puzzle information
        'category': category,
        'difficulty': difficulty,
        'question': question,

        #Category-specific information
        'content': content,

        #Answer information
        'answer_type': answer_type,
        'options': options if options is not None else [],
        'correct_answer': correct_answer,
        'accepted_answers': accepted_answers if accepted_answers is not None else [],

        #Hints and explanation
        'hints': hints,
        'explanation': explanation,

        #Puzzle metadata
        'source': source,
        'validation_status': 'pending'
    }

    return puzzle


In [39]:
required_fields = [
    "puzzle_id",
    "category",
    "difficulty",
    "question",
    "content",
    "answer_type",
    "options",
    "correct_answer",
    "accepted_answers",
    "hints",
    "explanation",
    "source",
    "validation_status"
]

for field in required_fields:
    assert field in number_puzzle, f"Missing field: {field}"

assert len(number_puzzle["hints"]) == 3, "A puzzle must have 3 hints."

print("Puzzle structure created successfully!")

Puzzle structure created successfully!


In [38]:
#common puzzle validation

def validate_common_fields(puzzle):
    errors = []

    required_fields = [
        "puzzle_id",
        "category",
        "difficulty",
        "question",
        "content",
        "answer_type",
        "options",
        "correct_answer",
        "accepted_answers",
        "hints",
        "explanation",
        "source",
        "validation_status"
    ]

    #Check required fields
    for field in required_fields:
        if field not in puzzle:
            errors.append(f"Missing required field: {field}")

    #Stop if required fields are missing
    if errors:
        return errors

    #Check category
    if puzzle['category'] not in PUZZLE_CATEGORIES:
        errors.append('Invalid puzzle category')

    #Check difficulty
    if puzzle['difficulty'] not in DIFFICULTY_LEVELS:
        errors.append('Invalid difficulty level')

    #Check question
    if not isinstance(puzzle['question'], str) or not puzzle['question'].strip():
        errors.append('Question must be a non-empty string')

    #Check hints
    if not isinstance(puzzle['hints'], list) or len(puzzle['hints'])!=3:
        errors.append('Each puzzle must contain exactly 3 hints')
    elif not all(isinstance(hint, str) and hint.strip() for hint in puzzle['hints']):
        errors.append('Every hint must be a non-empty string')

    #Check options
    if not isinstance(puzzle['options'], list):
        errors.append('Options must be a list')

    #Check explanation
    if not isinstance(puzzle['explanation'], str) or not puzzle['explanation'].strip():
        errors.append('Explanation must be a non-empty string')

    return errors

    

In [43]:
#validate number puzzle

def validate_number_pattern(puzzle):
    errors = []
    content = puzzle["content"]

    required_content_fields = [
        "sequence",
        "missing_position",
        "pattern_type"
    ]

    for field in required_content_fields:
        if field not in content:
            errors.append(f"Number pattern is missing: {field}")

    if errors:
        return errors

    sequence = content["sequence"]
    missing_position = content["missing_position"]

    if not isinstance(sequence, list) or len(sequence) < 4:
        errors.append("Sequence must contain at least 4 positions.")

    else:
        if not all(
            value is None or (
                isinstance(value, int) and not isinstance(value, bool)
            )
            for value in sequence
        ):
            errors.append("Sequence values must be integers or None.")

        if sequence.count(None) != 1:
            errors.append("Sequence must contain exactly one missing value.")

    if (
        not isinstance(missing_position, int)
        or isinstance(missing_position, bool)
        or not isinstance(sequence, list)
        or not (0 <= missing_position < len(sequence))
    ):
        errors.append("Missing position must be a valid zero-based index.")

    elif sequence[missing_position] is not None:
        errors.append("Specified missing position must contain None.")

    if not isinstance(content["pattern_type"], str) or not content["pattern_type"].strip():
        errors.append("Pattern type must be non-empty text.")

    if puzzle["answer_type"] != "integer":
        errors.append("Number pattern answer type must be integer.")

    answer = puzzle["correct_answer"]

    if not isinstance(answer, int) or isinstance(answer, bool):
        errors.append("Correct answer must be an integer.")

    options = puzzle["options"]

    if not isinstance(options, list) or len(options) != 4:
        errors.append("Number pattern must have exactly 4 options.")

    elif len(set(options)) != len(options):
        errors.append("Options must be unique.")

    elif answer not in options:
        errors.append("Correct answer must be included in the options.")

    if puzzle["accepted_answers"]:
        errors.append("Number pattern accepted_answers must be empty.")

    return errors

In [86]:
#riddles validation

def validate_riddle(puzzle):

    errors = []
    content = puzzle["content"]

    if "riddle_text" not in content:
        errors.append("Riddle is missing riddle_text.")

    elif (
        not isinstance(content["riddle_text"], str)
        or not content["riddle_text"].strip()
    ):
        errors.append("Riddle text must be non-empty.")

    if puzzle["answer_type"] != "text":
        errors.append("Riddle answer type must be text.")

    if (
        not isinstance(puzzle["correct_answer"], str)
        or not puzzle["correct_answer"].strip()
    ):
        errors.append("Correct answer must be non-empty text.")

    accepted_answers = puzzle["accepted_answers"]

    if not isinstance(accepted_answers, list):
        errors.append("Accepted answers must be a list.")

    elif not all(
        isinstance(answer, str) and answer.strip()
        for answer in accepted_answers
    ):
        errors.append("Every accepted answer must be non-empty text.")

    if puzzle["options"] != []:
        errors.append("Riddles should use free-text answers, not options.")

    return errors

In [11]:
#logic puzzle validation

def validate_logic(puzzle):
    errors=[]

    content = puzzle['content']

    required_content_fields = ['scenario', 'clues']

    for field in required_content_fields:
        if field not in content:
            errors.append(f"Logic puzzle is missing: {field}")

    if errors:
        return errors

    if not isinstance(content['scenario'], str) or not content['scenario'].strip():
        errors.append('Logic scenario must be non-empty text')

    clues = content['clues']

    if not isinstance(clues, list) or not clues:
        errors.append('Logic puzzle must contain at least one clue')

    elif not all (isinstance(clue, str) and clue.strip() for clue in clues):
        errors.append('Every logic clue must be non-empty text')

    if puzzle['answer_type'] not in ['text', 'structured']:
        errors.append('Logic answer type must be text or structured')

    return errors

In [12]:
#final validation function
def validate_puzzle(puzzle):
    errors = validate_common_fields(puzzle)

    if errors:
        return{
            'is_valid': False,
            'errors': errors
        }

    category = puzzle['category']

    if category == 'number_pattern':
        errors.extend(validate_number_pattern(puzzle))

    elif category == 'riddle':
        errors.extend(validate_riddle(puzzle))

    elif category == 'logic':
        errors.extend(validate_logic(puzzle))

    return{
        'is_valid': len(errors) ==0,
        'errors': errors
    }
    

In [13]:
#checking answers

def normalize_text(answer):
    #Standardize text answers before comparison
    if not isinstance(answer, str):
        return ""

    return " ".join(answer.strip().lower().split())

def check_answer(puzzle, player_answer):
    answer_type = puzzle['answer_type']
    correct_answer = puzzle['correct_answer']

    is_correct = False

    #Integer answers, used for number patterns
    if answer_type == 'integer':
        try:
            #Reject values such as 10.5 or ten
            if isinstance(player_answer, bool):
                raise ValueError

            submitted_answer = int(player_answer)

            if str(player_answer).strip() not in [str(submitted_answer), f"+{submitted_answer}"]:
                raise ValueError

            is_correct = submitted_answer == correct_answer

        except(ValueError, TypeError):
            is_correct = False

    #Text answers, used for riddles and some logic puzzles
    elif answer_type == 'text':
        normalized_answer = normalize_text(player_answer)

        accepted_answers = [
            correct_answer
        ] + puzzle.get('accepted_answers', [])

        normalized_accepted = [
            normalize_text(answer)
            for answer in accepted_answers
            if isinstance(answer, str)
        ]

        is_correct = normalized_answer in normalized_accepted

    #Structured answers, used for more complex logic puzzles
    elif answer_type == 'structured':
        is_correct = player_answer == correct_answer

    else: 
        raise ValueError(f"Unsupported answer type: {answer_type}")

    return {
        "is_correct": is_correct,
        "player_answer": player_answer,
        "feedback":(
            "Correct! Well done!"
            if is_correct
            else "Not quite. Try again!"
        ),
        "explanation": puzzle['explanation'] if is_correct else None
    }

In [14]:
number_puzzle = create_puzzle(
    category="number_pattern",
    difficulty="easy",

    question="Find the missing number in the sequence.",

    content={
        "sequence": [2, 4, 6, 8, None],
        "missing_position": 4,
        "pattern_type": "addition"
    },

    answer_type="integer",
    correct_answer=10,

    hints=[
        "Look carefully at how the numbers change.",
        "Compare each number with the number before it.",
        "Each number increases by the same amount."
    ],

    explanation="Each number increases by 2, so the missing number is 10.",

    options=[9, 10, 12, 14],

    source="curated"
)


riddle_puzzle = create_puzzle(
    category="riddle",
    difficulty="easy",
    question="What has cities but no houses, forests but no trees, and rivers but no water?",
    content={
        "riddle_text": "What has cities but no houses, forests but no trees, and rivers but no water?"
    },
    answer_type="text",
    correct_answer="map",
    accepted_answers=["a map"],
    hints=[
        "You might use it when travelling.",
        "It helps you find places.",
        "It shows roads, cities, and rivers."
    ],
    explanation="A map shows cities, forests, and rivers without containing the actual things.",
    source="curated"
)

logic_puzzle = create_puzzle(
    category="logic",
    difficulty="easy",
    question="Which person owns the dog?",
    content={
        "scenario": "Three friends own different pets.",
        "clues": [
            "Anna does not own a cat.",
            "Ben owns a dog.",
            "Cara does not own a dog."
        ]
    },
    answer_type="structured",
    correct_answer={"pet_owner": "Ben"},
    hints=[
        "Look at the clues carefully.",
        "One clue directly identifies the dog owner.",
        "Check what the second clue says about Ben."
    ],
    explanation="The second clue tells us that Ben owns the dog.",
    source="curated"
)



In [15]:
{
    "puzzle_id": "puzzle_123",
    "category": "number_pattern",
    "difficulty": "easy",
    "player_answer": 10,
    "is_correct": True,
    "time_taken": 25,
    "attempts": 1,
    "hints_used": 0,
    "timestamp": "..."
}

{'puzzle_id': 'puzzle_123',
 'category': 'number_pattern',
 'difficulty': 'easy',
 'player_answer': 10,
 'is_correct': True,
 'time_taken': 25,
 'attempts': 1,
 'hints_used': 0,
 'timestamp': '...'}

In [16]:
#record attempts

#attempt_history: stores detailed information about every puzzle attempt
#session statistics: provide a quick overview without having to recalculate everything
def record_attempt(
    session,
    puzzle,
    player_answer,
    is_correct,
    time_taken,
    attempts,
    hints_used
):
    attempt_record = {
        "attempt_id": str(uuid.uuid4()),
        "puzzle_id": puzzle['puzzle_id'],
        "category": puzzle['category'],
        "difficulty": puzzle['difficulty'],
        "player_answer": player_answer,
        "is_correct": is_correct,
        "time_taken": time_taken,
        "attempts": attempts,
        "hints_used": hints_used,
        "timestamp": datetime.now(timezone.utc).isoformat()
    }

    #Store individual attempt
    session['attempt_history'].append(attempt_record)

    #Update overall session statistics
    session['total_attempts'] += 1
    session['total_time'] += time_taken
    session['total_hints_used'] += hints_used

    if is_correct:
        session['correct_answers'] += 1
    else:
        session['incorrect_answers'] += 1

    return attempt_record

In [17]:
#a function to summarize the performance
def get_player_performance(session):
    total = session['total_attempts']
    correct = session['correct_answers']

    accuracy = (
        (correct/total)*100
        if total > 0 else 0
    )

    average_time = (
        session['total_time']/total
        if total > 0 else 0
    )

    average_hints = (
        session['total_hints_used']/total
        if total > 0 else 0
    )

    return {
        "total_attempts": total,
        "correct_answers": correct,
        "incorrect_answers": session['incorrect_answers'],
        "accuracy_percentage": round(accuracy, 2),
        "average_time_seconds": round(average_time, 2),
        "average_hints_used": round(average_hints, 2)
    }

In [18]:
#track recent performance
def get_recent_performance(session, limit=5):
    recent_attempts = session['attempt_history'][-limit:]

    if not recent_attempts:
        return{
            "attempts": [],
            "accuracy_percentage": 0
        }

    correct_count = sum(
        1 for attempt in recent_attempts
        if attempt['is_correct']
    )

    accuracy = (
        correct_count / len(recent_attempts)
    ) *100

    return {
        "attempts": recent_attempts,
        "accuracy_percentage": round(accuracy,2)
    }

In [19]:
test_session = create_game_session()

# Simulate three puzzle attempts
record_attempt(
    session=test_session,
    puzzle=number_puzzle,
    player_answer=10,
    is_correct=True,
    time_taken=25,
    attempts=1,
    hints_used=0
)

record_attempt(
    session=test_session,
    puzzle=number_puzzle,
    player_answer=12,
    is_correct=False,
    time_taken=40,
    attempts=1,
    hints_used=1
)

record_attempt(
    session=test_session,
    puzzle=number_puzzle,
    player_answer=10,
    is_correct=True,
    time_taken=30,
    attempts=2,
    hints_used=2
)

{'attempt_id': '4de62e05-af8e-4efd-834d-fe0e78814728',
 'puzzle_id': '04bbd277-5a0f-4626-bbb3-aab64d6dae1a',
 'category': 'number_pattern',
 'difficulty': 'easy',
 'player_answer': 10,
 'is_correct': True,
 'time_taken': 30,
 'attempts': 2,
 'hints_used': 2,
 'timestamp': '2026-11-18T19:41:33.325644+00:00'}

In [20]:
performance = get_player_performance(test_session)

print(json.dumps(performance, indent=4))

{
    "total_attempts": 3,
    "correct_answers": 2,
    "incorrect_answers": 1,
    "accuracy_percentage": 66.67,
    "average_time_seconds": 31.67,
    "average_hints_used": 1.0
}


In [21]:
recent = get_recent_performance(test_session)

print(json.dumps(recent, indent=4))

{
    "attempts": [
        {
            "attempt_id": "0a6c12cb-e7d1-44b4-9220-0e2b34a201e2",
            "puzzle_id": "04bbd277-5a0f-4626-bbb3-aab64d6dae1a",
            "category": "number_pattern",
            "difficulty": "easy",
            "player_answer": 10,
            "is_correct": true,
            "time_taken": 25,
            "attempts": 1,
            "hints_used": 0,
            "timestamp": "2026-11-18T19:41:33.325644+00:00"
        },
        {
            "attempt_id": "09d07a65-41d6-4705-80d9-0c9ec786761b",
            "puzzle_id": "04bbd277-5a0f-4626-bbb3-aab64d6dae1a",
            "category": "number_pattern",
            "difficulty": "easy",
            "player_answer": 12,
            "is_correct": false,
            "time_taken": 40,
            "attempts": 1,
            "hints_used": 1,
            "timestamp": "2026-11-18T19:41:33.325644+00:00"
        },
        {
            "attempt_id": "4de62e05-af8e-4efd-834d-fe0e78814728",
            "puzzle_id": 

In [22]:
#completed puzzles: the puzzles that the pplayer actually completes even after multiple attempts
def record_completed_puzzle(
    session,
    puzzle,
    final_answer,
    is_correct,
    time_taken,
    attempts,
    hints_used
):
    puzzle_record = {
        "puzzle_id": puzzle['puzzle_id'],
        "category": puzzle['category'],
        "difficulty": puzzle['difficulty'],
        "final_answer": final_answer,
        "is_correct": is_correct,
        "time_taken": time_taken,
        "attempts": attempts,
        "hints_used": hints_used,
        "completed_at": datetime.now(timezone.utc).isoformat()
    }

    session["puzzle_history"].append(puzzle_record)
    return puzzle_record

In [23]:
def build_adaptation_context(session, recent_limit=5):
    history = session['puzzle_history']
    recent_puzzles = history[-recent_limit:]
    total_completed = len(history)

    correct_count = sum(
        1 for puzzle in history
        if puzzle['is_correct']
    )

    accuracy = (
        correct_count / total_completed * 100
        if total_completed > 0
        else 0
    )

    recent_correct = sum(
        1 for puzzle in recent_puzzles
        if puzzle['is_correct']
    )

    recent_accuracy = (
        recent_correct/len(recent_puzzles) * 100
        if recent_puzzles
        else 0
    )

    return{
        "current_difficulty": session['current_difficulty'],
        "total_puzzles_completed": total_completed,
        "overall_accuracy_percentage": round(accuracy, 2),
        "recent_accuracy_percentage": round(recent_accuracy, 2),
        "recent_puzzles": recent_puzzles
    }

In [24]:
test_session = create_game_session()

record_completed_puzzle(
    session=test_session,
    puzzle=number_puzzle,
    final_answer=10,
    is_correct=True,
    time_taken=25,
    attempts=1,
    hints_used=0
)

record_completed_puzzle(
    session=test_session,
    puzzle=riddle_puzzle,
    final_answer="globe",
    is_correct=False,
    time_taken=55,
    attempts=2,
    hints_used=2
)

record_completed_puzzle(
    session=test_session,
    puzzle=logic_puzzle,
    final_answer={"pet_owner": "Ben"},
    is_correct=True,
    time_taken=35,
    attempts=1,
    hints_used=1
)

adaptation_context = build_adaptation_context(test_session)

print(json.dumps(adaptation_context, indent=4))

{
    "current_difficulty": "easy",
    "total_puzzles_completed": 3,
    "overall_accuracy_percentage": 66.67,
    "recent_accuracy_percentage": 66.67,
    "recent_puzzles": [
        {
            "puzzle_id": "04bbd277-5a0f-4626-bbb3-aab64d6dae1a",
            "category": "number_pattern",
            "difficulty": "easy",
            "final_answer": 10,
            "is_correct": true,
            "time_taken": 25,
            "attempts": 1,
            "hints_used": 0,
            "completed_at": "2026-11-18T19:41:42.490589+00:00"
        },
        {
            "puzzle_id": "c744148f-9e5b-49e9-a8b1-065c5b035fb3",
            "category": "riddle",
            "difficulty": "easy",
            "final_answer": "globe",
            "is_correct": false,
            "time_taken": 55,
            "attempts": 2,
            "hints_used": 2,
            "completed_at": "2026-11-18T19:41:42.490589+00:00"
        },
        {
            "puzzle_id": "e594fcb3-daec-4b4b-b9ae-abfb986128f6",


In [25]:
#adaptation prompt function
def get_adaptation_prompt(context):
    prompt = f"""
    You are the adaptive difficulty manager for Puzzle Pirates, an AI-ppowered brain puzzle game.
    Your job is to analyze player performance and recommend the difficulty of the next puzzle.

    Available difficulty levels:
    - easy
    - medium
    - hard

    Player performance data:
    {json.dumps(context, indent=2)}

    Decision guidelines:

    1. Consider both overall and recent performance.
    2. Consider correctness, completion time, attempts, and hints used.
    3. Consistently strong performance may justify increasing difficulty.
    4. Repeated mistakes or substantial use of hints may justify decreasing difficulty.
    5. Mixed performance may justify maintaining current difficulty.
    6. Do not make a large jump without sufficient evidence.
    7. If the player has completed very few puzzles, be conservative.

    Return ONLY valud JSON in this exact format:

    {{
        "recommended_difficulty":"easy",
        "decision":"maintain",
        "reasoning": "Brief explanation of your decision"
    }}

    The decision must be one of:
    - increase
    - decrease
    - maintain

    The recommended difficulty must be easy, medium, or hard.
    """

    return prompt

In [26]:
def get_ai_difficulty_recommendation(session):
    context = build_adaptation_context(session)

    prompt = get_adaptation_prompt(context)

    response = client.models.generate_content(
        model = MODEL_NAME,
        contents = prompt,
        config={
            "response_mime_type": "application/json"
        }
    )

    recommendation = json.loads(response.text)

    #Validate the AI response
    recommended_difficulty = recommendation.get("recommended_difficulty")

    if recommended_difficulty not in DIFFICULTY_LEVELS:
        raise ValueError("Gemini returned an invalid difficulty")

    if recommendation.get("decision") not in ["increase", "decrease", "maintain"]:
        raise ValueWrror("Gemini returned an invalid decision")

    if not recommendation.get('reasoning'):
        raise ValueError("Gemini did not provide reasoning")

    return recommendation
    

In [27]:
#update the game session
def update_session_difficulty(session, recommendation):
    previous_difficulty = session['current_difficulty']
    new_difficulty = recommendation['recommended_difficulty']

    session['current_difficulty'] = new_difficulty

    adaptation_record = {
        "previous_difficulty": previous_difficulty,
        "new_difficulty": new_difficulty,
        "decision": recommendation['decision'],
        "reasoning": recommendation['reasoning'],
        "timestamp": datetime.now(timezone.utc).isoformat()
    }

    session['adaptation_history'].append(adaptation_record)

    return adaptation_record

In [28]:
recommendation = get_ai_difficulty_recommendation(test_session)

print("AI recommendation:")
print(json.dumps(recommendation, indent=4))

adaptation_result = update_session_difficulty(
    test_session,
    recommendation
)

print("\nUpdated session difficulty:")
print(test_session["current_difficulty"])

print("\nAdaptation record:")
print(json.dumps(adaptation_result, indent=4))

AI recommendation:
{
    "recommended_difficulty": "easy",
    "decision": "maintain",
    "reasoning": "The player has completed only a few puzzles with a mixed recent accuracy of 66.67%, including incorrect answers and hint usage on easy difficulty. A conservative approach is warranted before increasing the difficulty."
}

Updated session difficulty:
easy

Adaptation record:
{
    "previous_difficulty": "easy",
    "new_difficulty": "easy",
    "decision": "maintain",
    "reasoning": "The player has completed only a few puzzles with a mixed recent accuracy of 66.67%, including incorrect answers and hint usage on easy difficulty. A conservative approach is warranted before increasing the difficulty.",
    "timestamp": "2026-11-18T19:41:48.118136+00:00"
}


In [75]:
#generate number puzzle prompt

def get_number_pattern_prompt(difficulty, recent_puzzles=None, feedback=None):

    recent_puzzles = recent_puzzles or []

    DIFFICULTY_GUIDELINES = {
    "easy": """
    Use straightforward patterns such as constant addition,
    subtraction, multiplication, or division.
    """,

    "medium": """
    Use moderately challenging patterns such as alternating
    operations, increasing differences, or combinations of
    simple operations.
    """,

    "hard": """
    Use challenging patterns such as multiple interacting
    sequences, complex difference patterns, or less obvious
    mathematical relationships.
    """
    }

    prompt = f"""
    You are an expert puzzle creator for Puzzle Pirates, an adaptive brain puzzle game.
    
    Generate ONE original number pattern puzzle.
    
    DIFFICULTY: {difficulty}
    difficulty_guidelines = {DIFFICULTY_GUIDELINES[difficulty]}
    
    PUZZLE REQUIREMENTS:
    
    1. The puzzle must have exactly one missing number.
    2. The sequence must contain at least 4 numbers.
    3. The missing number must be represented as null.
    4. Use a clear and mathematically consistent pattern.
    5. The correct answer must be an integer.
    6. Provide exactly 4 multiple-choice options.
    7. Only one option can be correct.
    8. Provide exactly 3 progressive hints.
    9. Hints must gradually reveal the solving approach without immediately giving away the answer.
    10. Provide a clear explanation of the solution.
    11. Avoid ambiguous sequences with multiple valid answers.
    12. Match the complexity of the requested difficulty.
    
    VARIETY REQUIREMENTS:
    
    - Use different mathematical patterns.
    - Vary the starting numbers and sequence length.
    - Explore addition, subtraction, multiplication, division, alternating patterns, and other valid patterns.
    - Avoid repeatedly generating simple arithmetic sequences.
    - Do not copy the examples or previously generated puzzles.
    
    RECENTLY GENERATED PUZZLES TO AVOID:
    
    {json.dumps(recent_puzzles, indent=2)}
    
    Generate something substantially different from these puzzles.
    
    Return ONLY valid JSON using this structure:
    {{
        "category": "number_pattern",
        "difficulty": "{difficulty}",
        "question": "Find the missing number.",
        "content": {{
            "sequence": [3, 6, 9, null, 15],
            "missing_position": 3,
            "pattern_type": "addition"
        }},
        "answer_type": "integer",
        "correct_answer": 12,
        "options": [10, 11, 12, 14],
        "accepted_answers": [],
        "hints": [
            "Hint 1",
            "Hint 2",
            "Hint 3"
        ],
        "explanation": "Explain the pattern and answer."
    }}

    The example above is only a format reference. Generate a new puzzle rather than copying it.
    """
    
    if feedback:
        prompt += f"""
    
    PREVIOUS VALIDATION ERRORS:
    {feedback}
    
    Correct these issues in the new puzzle.
    """

    prompt += "\nReturn only valid JSON. Do not include Markdown."

    return prompt

In [62]:
def generate_number_pattern(difficulty, feedback=None):

    recent_puzzles = generated_puzzle_history[-10:]
    
    prompt = get_number_pattern_prompt(
        difficulty,
        recent_puzzles = recent_puzzles,
        feedback=feedback
    )

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config={
            "response_mime_type": "application/json",
            "temperature": 0.9
        }
    )

    if not response.text:
        raise ValueError("Gemini returned an empty response.")

    return json.loads(response.text)

In [33]:
#convert the AI response to our common puzzle structure using the create_puzzle function

def convert_to_puzzle(generated_data):

    puzzle = create_puzzle(
        category = generated_data['category'],
        difficulty = generated_data['difficulty'],
        question = generated_data['question'],
        content = generated_data['content'],
        answer_type = generated_data['answer_type'],
        correct_answer = generated_data['correct_answer'],
        hints = generated_data['hints'],
        explanation = generated_data['explanation'],
        options = generated_data['options'],
        accepted_answers = generated_data['accepted_answers'],
        source = 'ai'
    )

    return puzzle

In [46]:
def generate_validated_number_pattern(
    difficulty,
    max_retries=3
):

    feedback = None

    for attempt in range(1, max_retries + 1):

        print(f"Generation attempt {attempt}/{max_retries}")

        try:
            # Generate puzzle using Gemini
            generated_data = generate_number_pattern(
                difficulty,
                feedback=feedback
            )

            # Convert into our common puzzle structure
            puzzle = convert_to_puzzle(generated_data)

            # Validate the puzzle
            validation_result = validate_puzzle(puzzle)

            if validation_result["is_valid"]:

                puzzle["validation_status"] = "passed"

                print("Puzzle generated and validated successfully!")

                return puzzle

            # Prepare feedback for the next generation attempt
            feedback = "\n".join(
                validation_result["errors"]
            )

            print("Validation failed:")

            for error in validation_result["errors"]:
                print("-", error)

        except (ValueError, KeyError, TypeError, json.JSONDecodeError) as error:

            feedback = str(error)

            print("Generation or parsing error:", error)

    raise RuntimeError(
        f"Could not generate a valid puzzle after {max_retries} attempts."
    )

In [69]:
generated_puzzle_history = []


In [78]:
# for i in range(3):

#     generated_data = generate_number_pattern("hard")

#     print(f"\nPuzzle {i + 1}")
#     print("Sequence:", generated_data["content"]["sequence"])
#     print("Pattern:", generated_data["content"]["pattern_type"])
#     print("Answer:", generated_data["correct_answer"])

#     puzzle = convert_to_puzzle(generated_data)

#     result = validate_puzzle(puzzle)

#     if result["is_valid"]:
#         generated_puzzle_history.append({
#             "sequence": puzzle["content"]["sequence"],
#             "pattern_type": puzzle["content"]["pattern_type"],
#             "correct_answer": puzzle["correct_answer"]
#         })

In [79]:
#riddle generator

def get_riddle_prompt(difficulty, recent_puzzles=None, feedback=None):

    recent_puzzles = recent_puzzles or []

    difficulty_guidelines = {
        "easy": """
            Use familiar objects, everyday situations, and straightforward clues. The answer should be
            relatively easy to infer.
        """,
        "medium": """
            Use more indirect clues, wordplay, and multiple reasoning steps. The answer should require
            some thought.
        """,
        "hard": """
            Use challenging wordplay, layered clues, and less obvious connections. Avoid obscure trivia
            or knowledge that players would need to look up.
        """
    }

    if difficulty not in difficulty_guidelines:
        raise ValueError("Invalid difficulty level.")

    prompt = f"""
    Your are an expert riddle creator for Puzzle Pirated, an adaptive brain puzzle game.

    Generate ONE original riddle.

    DIFFICULTY: {difficulty}
    difficulty_guidelines = {difficulty_guidelines[difficulty]}

    PUZZLE REQUIREMENTS:

    1. Generate one clear and engaging riddle.
    2. The riddle must have one intended answer.
    3. Avoid ambiguous clues that support multiple unrelated answers.
    4. Provide the correct answer as a short text string.
    5. Provide alternative accepted answers where appropriate.
    6. Do not include the correct answer in the question.
    7. Generate exactly 3 progressive hints.
    8. Each hint should reveal more information than the previous one.
    9. Hints must not immediately reveal the answer.
    10. Provide a clear explanation of the answer.
    11. Avoid obscure trivia and culturally specific references.
    12. Generate an original riddle rather than copying common examples.

    VARIETY REQUIREMENTS:

    - Explore different themes, objects, situations, and types of wordplay.
    - Avoid repeatedly using familiar riddles.
    - Avoid generating riddles similar to the recent examples below.

    RECENTLY GENERATED RIDDLES TO AVOID:
    {json.dumps(recent_puzzles, indent=2)}

    Return ONLY valid JSON with these fields:
    {{
        "category": "riddle",
        "difficulty": "{difficulty}",
        "question": "The complete riddle text",
        "content": {{
            "riddle_text": "The complete riddle text"
        }},
        "answer_type": "text",
        "correct_answer": "intended answer",
        "options": [],
        "accepted_answers": ["alternative answer"],
        "hints": [
            "First hint",
            "Second hint",
            "Third hint"
        ],
        "explanation": "Explain why this is the answer."
    }}

    Do not include Markdown or additional text.
    
    """

    if feedback:
        prompt += f"""
            PREVIOUS VALIDATION ERRORS: {feedback}

            Generate a new riddle that corrects these issues. 
        """

    return prompt

In [80]:
def generate_riddle(difficulty, recent_puzzles=None, feedback=None):
    prompt = get_riddle_prompt(
        difficulty,
        recent_puzzles=recent_puzzles,
        feedback=feedback
    )

    response = client.models.generate_content(
        model = MODEL_NAME,
        contents= prompt,
        config={
            "response_mime_type": "application/json",
            "temperature": 0.9
        }
    )

    if not response.text:
        raise ValueError('Gemini returned an empty response.')

    generated_data = json.loads(response.text)

    return generated_data

In [94]:
riddle_data = generate_riddle("easy")

print(json.dumps(riddle_data, indent=4))

{
    "category": "riddle",
    "difficulty": "easy",
    "question": "I have four legs, but I cannot walk. I hold your meals, but I do not eat. What am I?",
    "content": {
        "riddle_text": "I have four legs, but I cannot walk. I hold your meals, but I do not eat. What am I?"
    },
    "answer_type": "text",
    "correct_answer": "table",
    "options": [],
    "accepted_answers": [
        "a table"
    ],
    "hints": [
        "You often sit around me when it is time for dinner.",
        "I usually stand in the middle of a dining room or kitchen.",
        "People place plates, bowls, and cups on my flat top surface."
    ],
    "explanation": "A table has four legs for support, but it is a piece of furniture and cannot walk. Its main purpose is to hold food and dishes during meals."
}


In [85]:
riddle_puzzle = convert_to_puzzle(riddle_data)

print(json.dumps(riddle_puzzle, indent=4))

{
    "puzzle_id": "fb1cf9d6-3a36-4c53-93a1-c802e79e0912",
    "category": "riddle",
    "difficulty": "easy",
    "question": "I have four legs and a flat back, but I cannot walk or run. People pull up a seat to me three times a day to eat. What am I?",
    "content": {
        "riddle_text": "I have four legs and a flat back, but I cannot walk or run. People pull up a seat to me three times a day to eat. What am I?"
    },
    "answer_type": "text",
    "options": [],
    "correct_answer": "table",
    "accepted_answers": [
        "dining table",
        "kitchen table"
    ],
    "hints": [
        "I stay in the dining room or kitchen.",
        "Dishes, bowls, and food are placed directly on top of me.",
        "You sit around me with your family for meals."
    ],
    "explanation": "A table has four legs and a flat surface top, and it is the primary furniture around which people gather to eat meals.",
    "source": "ai",
    "validation_status": "pending"
}


In [87]:
validation_result = validate_puzzle(riddle_puzzle)

print(json.dumps(validation_result, indent=4))

{
    "is_valid": true,
    "errors": []
}


In [88]:
if validation_result["is_valid"]:

    riddle_puzzle["validation_status"] = "passed"

    print("Riddle generated and validated successfully!")

else:

    riddle_puzzle["validation_status"] = "failed"

    for error in validation_result["errors"]:
        print("-", error)

Riddle generated and validated successfully!


In [89]:
print("QUESTION:")
print(riddle_puzzle["question"])

print("\nHINTS:")

for i, hint in enumerate(riddle_puzzle["hints"], start=1):
    print(f"{i}. {hint}")

print("\nCORRECT ANSWER:")
print(riddle_puzzle["correct_answer"])

print("\nACCEPTED ALTERNATIVES:")
print(riddle_puzzle["accepted_answers"])

print("\nEXPLANATION:")
print(riddle_puzzle["explanation"])

QUESTION:
I have four legs and a flat back, but I cannot walk or run. People pull up a seat to me three times a day to eat. What am I?

HINTS:
1. I stay in the dining room or kitchen.
2. Dishes, bowls, and food are placed directly on top of me.
3. You sit around me with your family for meals.

CORRECT ANSWER:
table

ACCEPTED ALTERNATIVES:
['dining table', 'kitchen table']

EXPLANATION:
A table has four legs and a flat surface top, and it is the primary furniture around which people gather to eat meals.


In [101]:
def get_logic_puzzle_prompt(difficulty, recent_puzzles=None, feedback=None):
    recent_puzzles = recent_puzzles or []

    difficulty_guidelines = {
        "easy": """
        Generate a simple puzzle with 3 people and 3 items. Use straightforward clues and simple 
        elimination. The player should be able to solve it with a few deductions.
        """,

        "medium": """
        Generate a puzzle with 3 to 4 entities in each group. Use a combination of direct and indirect
        clues. Require multiple connected deductions.
        """,

        "hard": """
        Generate a puzzle with 4 to 5 entities in each group. Use more interconnected clues and 
        multiple reasoning steps. Avoid making the puzzle unnecessarily complicated.
        """
    }

    if difficulty not in difficulty_guidelines:
        raise ValueError('Invalid difficulty level.')

    prompt = f"""
    Your are an expert logic puzzle creator for Puzzle Pirated, an adaptive brain puzzle game.

    Generate ONE original matching and deduction logic puzzle.

    DIFFICULTY: {difficulty}
    DIFFICULTY EXPECTATIONS: {difficulty_guidelines[difficulty]}

    PUZZLE REQUIREMENTS:

    1. Create two groups of entities.
    2. Each group must contain the same number of unique entities.
    3. Each entity in the first group must match exactly one entity in the second group. 
    4. Every entity in the second group must be matched exactly once.
    5. Provide a scenario that explains the puzzle.
    6. Provide clear clues that allow the player to determine every match. 
    7. The clues must be logically consistent.
    8. There must be exactly ONE valid solution.
    9. The correct answer must be a complete mapping.
    10. Generate exactly 3 progressive hints.
    11. Hints should guide the player's reasoning without directly revealing the complete solution.
    12. Include a clear explanation of the solution.
    13. Avoid trivia or outside knowledge.
    14. Make the puzzle original and avoid repeating recent puzzles.

    VARIETY REQUIREMENTS:

    - Explore different themes and scenarios.
    - Vary the names and entities.
    - Vary the types of clues and deduction steps.
    - Avoid copying recently generated puzzles.

    RECENTLY GENERATED PUZZLES TO AVOID:
    {json.dumps(recent_puzzles, indent=2)}

    Return ONLY valid JSON using these fields:
    {{
        "category": "logic",
        "difficulty": "{difficulty}",
        "question": "The puzzle question shown to the player.",
        "content": {{
            "scenario": "Scenario description.",
            "left_group": ["Person A", "Person B", "Person C"],
            "right_group": ["Item A", "Item B", "Item C"],
            "clues": [
                "First clue",
                "Second clue",
                "Third clue"
            ]
        }},
        "answer_type": "structured",
        "correct_answer": {{
            "Person A": "Item B",
            "Person B": "Item C",
            "Person C": "Item A"
        }},
        "options": [],
        "accepted_answers": [],
        "hints": [
            "First hint",
            "Second hint",
            "Third hint"
        ],
        "explanation": "Explain the deductions leading to the solution."
    }}

    The example values illustrate the required structure only.
    Generate a new puzzle and a logically consistent solution. 

    Return only valid JSON. Do not include Markdown. 
    
    """

    if feedback:
        prompt += f"""

        PREVIOUS VALIDATION ERRORS: {feedback}

        Generate a new puzzle that corrects these issues.
        """

    return prompt

In [97]:
def generate_logic_puzzle(difficulty, recent_puzzles=None, feedback=None):
    prompt = get_logic_puzzle_prompt(difficulty, 
                                    recent_puzzles=recent_puzzles,
                                    feedback=feedback)

    response = client.models.generate_content(
        model = MODEL_NAME,
        contents = prompt,
        config = {
            "response_mime_type": "application/json",
            "temperature": 0.9
        }
    )

    if not response.text:
        raise ValueError('Gemini returned an empty response')

    generated_data = json.loads(response.text)

    return generated_data

In [103]:
logic_data = generate_logic_puzzle("easy")

print(json.dumps(logic_data, indent=4))

{
    "category": "logic",
    "difficulty": "easy",
    "question": "Determine which explorer brought which magical artifact back from the Whispering Caves.",
    "content": {
        "scenario": "Three brave explorers\u2014Lyra, Jax, and Orion\u2014each emerged from the Whispering Caves carrying a unique magical artifact: a glowing crystal, a whispering shell, and an ancient scroll.",
        "left_group": [
            "Lyra",
            "Jax",
            "Orion"
        ],
        "right_group": [
            "Glowing Crystal",
            "Whispering Shell",
            "Ancient Scroll"
        ],
        "clues": [
            "Lyra did not bring back the Glowing Crystal.",
            "The explorer who brought the Ancient Scroll is not Orion.",
            "Jax carried the Whispering Shell."
        ]
    },
    "answer_type": "structured",
    "correct_answer": {
        "Lyra": "Ancient Scroll",
        "Jax": "Whispering Shell",
        "Orion": "Glowing Crystal"
    },
   